# Apple Health Data Analysis

This notebook demonstrates the health data analysis workflow for parsing,
cleaning, and analyzing ECG and FHIR clinical records from an Apple Health export.

## Setup

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

# Define paths
BASE_DIR = Path.cwd().parent
ECG_DIR = BASE_DIR / "electrocardiograms"
FHIR_DIR = BASE_DIR / "clinical-records"
OUTPUT_DIR = BASE_DIR / "output"

## Load Analysis Modules

In [ ]:
from analysis.ecg_parser import aggregate_all_ecg, parse_ecg_full
from analysis.fhir_parser import aggregate_all_fhir, get_resource_type_counts
from analysis.data_munging import (
    convert_datetime_columns,
    standardize_column_names,
    handle_missing_values,
)
from analysis.summary import (
    get_ecg_summary,
    get_fhir_summary,
    create_timeline_summary,
)

## ECG Data Analysis

In [ ]:
# Load and aggregate all ECG data
ecg_df = aggregate_all_ecg(ECG_DIR)
print(f"Loaded {len(ecg_df):,} ECG samples from {ecg_df['source_file'].nunique()} recordings")

# Standardize column names
ecg_df = standardize_column_names(ecg_df, {
    'lead_i_mv': 'lead_i_microvolts',
    'recorded_date': 'ecg_datetime',
})

# Convert datetime columns
ecg_df = convert_datetime_columns(ecg_df, ['ecg_datetime', 'date_of_birth'])

# Display sample
ecg_df.head()

In [ ]:
# ECG summary statistics
ecg_summary = get_ecg_summary(ecg_df)
print("\n=== ECG Summary ===")
print(f"Total recordings: {ecg_summary['total_recordings']}")
print(f"Total samples: {ecg_summary['total_samples']:,}")
print(f"Date range: {ecg_summary['earliest_recording']} to {ecg_summary['latest_recording']}")
print(f"\nClassifications:")
for cls, count in ecg_summary.get('classifications', {}).items():
    print(f"  {cls}: {count}")
print(f"\nLead I amplitude stats:")
for stat, val in ecg_summary['lead_i_stats'].items():
    print(f"  {stat}: {val:.2f}")

In [ ]:
# Plot ECG waveform from a single recording
sample_file = ecg_df['source_file'].iloc[0]
sample_ecg = ecg_df[ecg_df['source_file'] == sample_file].copy()

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(sample_ecg['timestamp_ms'] / 1000, sample_ecg['lead_i_microvolts'], 
        linewidth=0.5, color='blue')
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Lead I (µV)')
ax.set_title(f'ECG Waveform - {sample_file}')
plt.tight_layout()
plt.show()

In [ ]:
# Recording frequency over time
ecg_df['year'] = ecg_df['ecg_datetime'].dt.year
ecg_by_year = ecg_df.groupby('year')['source_file'].nunique()

fig, ax = plt.subplots(figsize=(10, 4))
ecg_by_year.plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Year')
ax.set_ylabel('Number of ECG Recordings')
ax.set_title('ECG Recordings by Year')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## FHIR Clinical Records Analysis

In [ ]:
# Load and aggregate all FHIR data
fhir_df = aggregate_all_fhir(FHIR_DIR)
print(f"Loaded {len(fhir_df):,} FHIR records")

# Standardize column names
fhir_df = standardize_column_names(fhir_df, {
    'effective_datetime': 'observation_datetime',
})

# Convert datetime columns
fhir_df = convert_datetime_columns(fhir_df, [
    'observation_datetime', 'condition_onset_datetime',
    'occurrence_datetime', 'recorded_date', 'issued_datetime',
])

# Handle missing values
fhir_df = handle_missing_values(fhir_df, strategy='flag')

# Display resource type distribution
print("\nRecords by type:")
print(fhir_df['resource_type'].value_counts())

In [ ]:
# FHIR summary
fhir_summary = get_fhir_summary(fhir_df)
print("\n=== FHIR Summary ===")
print(f"Total records: {fhir_summary['total_records']:,}")

if 'condition_count' in fhir_summary:
    print(f"\nConditions: {fhir_summary['condition_count']}")
    print("Top conditions:")
    for cond, count in list(fhir_summary.get('top_conditions', {}).items())[:5]:
        print(f"  {cond}: {count}")

if 'immunization_count' in fhir_summary:
    print(f"\nImmunizations: {fhir_summary['immunization_count']}")
    print("Vaccines given:")
    for vaccine, count in fhir_summary.get('vaccines_given', {}).items():
        print(f"  {vaccine}: {count}")

if 'allergy_count' in fhir_summary:
    print(f"\nAllergies: {fhir_summary['allergy_count']}")
    for allergy, count in fhir_summary.get('allergies', {}).items():
        print(f"  {allergy}: {count}")

In [ ]:
# Create timeline of FHIR records
timeline = create_timeline_summary(fhir_df, 'observation_datetime')
timeline.head(10)

In [ ]:
# Plot timeline
if 'Observation' in timeline.columns:
    fig, ax = plt.subplots(figsize=(14, 5))
    timeline.plot(x='year_month', y='Observation', kind='bar', ax=ax, 
                  color='forestgreen', alpha=0.7)
    ax.set_xlabel('Year-Month')
    ax.set_ylabel('Number of Observations')
    ax.set_title('Observations Over Time')
    # Show only some x-tick labels
    tick_positions = range(0, len(timeline), max(1, len(timeline) // 12))
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([timeline['year_month'].iloc[i] for i in tick_positions], 
                      rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Data Quality Assessment

In [ ]:
# Missing value analysis
missing_by_type = fhir_df.groupby('resource_type').apply(
    lambda x: (x.isnull().sum() / len(x) * 100).round(1)
).mean(axis=1).sort_values(ascending=False)

print("Average missing percentage by resource type:")
print(missing_by_type)

In [ ]:
# Observations with values - extract vital signs
observations = fhir_df[fhir_df['resource_type'] == 'Observation'].copy()
vital_signs = observations[observations['component_display'].notna()]

# Summary of vital sign measurements
vital_summary = vital_signs.groupby('component_display')['value'].agg(
    ['count', 'mean', 'min', 'max']
).round(2)

vital_summary

In [ ]:
# Plot vital sign trends over time
if len(vital_signs) > 0:
    vital_signs['year_month'] = vital_signs['observation_datetime'].dt.to_period('M')
    
    # Blood pressure trend
    bp = vital_signs[vital_signs['component_display'] == 'Systolic blood pressure']
    if len(bp) > 0:
        bp_monthly = bp.groupby('year_month')['value'].mean()
        
        fig, ax = plt.subplots(figsize=(12, 4))
        bp_monthly.plot(ax=ax, marker='o', markersize=3, color='crimson')
        ax.set_xlabel('Date')
        ax.set_ylabel('Systolic BP (mmHg)')
        ax.set_title('Systolic Blood Pressure Trend')
        ax.axhline(y=120, color='green', linestyle='--', alpha=0.5, label='Normal threshold')
        ax.axhline(y=140, color='orange', linestyle='--', alpha=0.5, label='Hypertension threshold')
        ax.legend()
        plt.tight_layout()
        plt.show()

## Save Processed Data

In [ ]:
# Export processed data
OUTPUT_DIR.mkdir(exist_ok=True)

# Save ECG data
ecg_df.to_csv(OUTPUT_DIR / 'ecg_processed.csv', index=False)
print(f"Saved ECG data to {OUTPUT_DIR / 'ecg_processed.csv'}")

# Save FHIR data
fhir_df.to_csv(OUTPUT_DIR / 'fhir_processed.csv', index=False)
print(f"Saved FHIR data to {OUTPUT_DIR / 'fhir_processed.csv'}")

# Save summaries as JSON
import json

with open(OUTPUT_DIR / 'ecg_summary.json', 'w') as f:
    json.dump({k: str(v) if hasattr(v, 'strftime') else v 
               for k, v in ecg_summary.items()}, f, indent=2, default=str)

with open(OUTPUT_DIR / 'fhir_summary.json', 'w') as f:
    json.dump(fhir_summary, f, indent=2, default=str)

print(f"Saved summaries to {OUTPUT_DIR}")